# Wave Gate Check

Deterministic gate for the build waves job. Runs after each `wN_integrate` Genie Code task.

A Genie Code task can finish "successfully" even when it concluded the wave failed, so this notebook reads the integrator's gate file and sets a `gate_passed` **task value**. A downstream `If/else` condition task checks the value — if false, all remaining wave tasks are `Excluded` (zero compute on cascade).

Gate file: `<build_root>/<status_dir>/<run_id>/wave_<wave>_gate.json` (written by the wave integrator skill).

**Runtime gate.** If the wave lists `runtime_gate` targets, this notebook also checks each one itself (it does not trust Genie Code reports):
* **app**: active deployment `SUCCEEDED` and created after the build run started, app `RUNNING` / compute `ACTIVE`, unchanged through `soak_minutes`, OTel error lines for the deployment ≤ `max_errors`, all `required_markers` present. With `probes`: each HTTP probe (or MCP `initialize`) passes at the start of the soak, authenticated as the `probe_auth` service principal (OAuth M2M, credentials from a secret scope; one retry each; `app_probes.py`). With `stability.min_traces` / `max_trace_errors`: enough traces and no more error traces than allowed since the probes ran, in each trace source: OTel spans in `telemetry.traces_table` (SERVER spans counted by default, errors = `STATUS_CODE_ERROR`) and MLflow 3 traces in `telemetry.mlflow_experiment_id` (errors = state `ERROR`). Waits up to `traces_wait_minutes` for ingestion.
* **job**: latest run since the build run started finished `SUCCESS`.
* **pipeline**: latest update since the build run started `COMPLETED` (or stays `RUNNING` through the soak). With `stability.max_failed_records` / `max_dropped_records`, that update's expectation metrics (from the pipeline's UC event log table if it publishes one, else the events API) must stay within the limits; failing expectations are listed by name.
* **dashboard**: published revision created after the build run started, a warehouse is set, and every dataset resolves: `EXPLAIN` by default (a plan containing `Error occurred during query planning` fails), or `SELECT * FROM (<query>) LIMIT 1` with `execute_datasets: true`. Table / metric view datasets must resolve in UC.
* **genie_space** (static checks first): freshness = space updated after the build run started (`update_time`), **or** its deployed content equals the bundle source (`source_file`, else `resource_key` looked up in the bundle YAML; normalised: line arrays joined, `metric_views` merged into `tables`, object lists sorted), so a no-op deploy passes as `current`; warehouse attached and exists (and matches `warehouse_id` if set), every table / metric view resolves in UC, every `column_configs` column still exists, every example SQL passes `EXPLAIN` (`check_example_sql`, default on). Then, if set (and even when only freshness failed, as advisory results), each of `probe_questions` is asked in a new conversation (one retry): pass = Genie produced SQL and it ran. Answers are never judged. `probe_mode: warn` (default) records failures as `warnings` without failing the wave; `fail` fails it.
* **table** (tables, views, materialized views, streaming tables, metric views): resolves in UC, has every `required_columns`, at least `min_rows` rows (`COUNT(*)`), and was written during this run: last Delta commit (`DESCRIBE HISTORY`) or, for materialized views, last refresh. Views, metric views and foreign tables have no data commits, so freshness is skipped for them. No soak, no deploy.

**Declared bundle validations.** Every `validate_bundles` entry of the wave must appear in the gate file's `bundle_validations` with `result: pass`.

**Required checks.** Every name in the wave's `required_checks` must appear in the gate file's `checks` with `result: pass` (missing or `skipped` fails).

**Session outcomes.** The gate reads each session iteration's agent response and conversation id from the Jobs API (`runs/get` + `runs/get-output`). A session that hit the Genie Code step limit, is still `in_progress`, or wrote no status file is reported by name with its conversation id and the next step. Jobs API errors are warnings; the status-file checks still apply.

The runtime verdict is written to `wave_<wave>_runtime.json` next to the gate file. Runtime checks are skipped if the static gate already failed.

**Result and run verdict.** The gate never raises on a failed wave: raising would mark the next wave `UPSTREAM_FAILED`, and its `run_if: ALL_DONE` integrator and gate would still run. It prints `GATE_FAILED: wave <wave>: ...` and the result JSON, writes `wave_<wave>_gate_result.json`, and sets `gate_passed`. The final `build_verdict` task reads every gate's result and fails the run.

In [0]:
%pip install --upgrade databricks-sdk mlflow
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("wave", "", "Wave id")
dbutils.widgets.text("run_id", "", "Job run id")
dbutils.widgets.text("build_root", "", "Build root (project Git folder)")
dbutils.widgets.text("manifest_path", "", "Manifest path (waves.json)")

wave = dbutils.widgets.get("wave").strip()
run_id = dbutils.widgets.get("run_id").strip()
build_root = dbutils.widgets.get("build_root").strip()
manifest_path = dbutils.widgets.get("manifest_path").strip()

if not wave or not run_id:
    raise ValueError(f"wave and run_id are required (got wave={wave!r}, run_id={run_id!r})")

print(f"wave={wave} run_id={run_id}")

In [0]:
import json
import os

with open(manifest_path) as f:
    manifest = json.load(f)

# waves_manifest.py sits next to this notebook (cwd for workspace notebooks); shared with setup.
import sys
sys.path.insert(0, os.getcwd())
from waves_manifest import is_archived, last_runnable_wave_id
from session_status import (check_files_changed, iteration_index, latest_by_index, prior_run_files,
                            session_outcome, session_problem)
from gate_helpers import gate_failure_message, gate_result_path, required_check_problems
from jobs_api import get_run, get_run_output
from databricks.sdk import WorkspaceClient

wave_cfg = next((w for w in manifest["waves"] if str(w["wave_id"]) == wave), None)
if wave_cfg is None:
    raise ValueError(f"Wave {wave!r} not found in {manifest_path}")
if is_archived(wave_cfg):
    # Archived waves get no tasks; their sessions never write status files in this build.
    raise ValueError(f"Wave {wave!r} is archived; re-run setup so the build job has no tasks for it")

status_dir = os.path.join(build_root, manifest["status_dir"], run_id)
problems = []

# 1. Integrator gate file
gate_path = os.path.join(status_dir, f"wave_{wave}_gate.json")
gate = None
if not os.path.exists(gate_path):
    problems.append(f"gate file missing: {gate_path}")
else:
    with open(gate_path) as f:
        gate = json.load(f)
    if gate.get("status") != "pass":
        problems.append(f"gate status={gate.get('status')!r}: {gate.get('reason', '')}")
    if gate.get("stray_files"):
        problems.append(f"stray files: {gate['stray_files']}")
    # Carryover from earlier runs is tolerated, except code nobody claimed by the last runnable wave.
    if str(last_runnable_wave_id(manifest)) == wave:
        unclaimed = [c.get("path") for c in (gate.get("carryover_files") or [])
                     if c.get("kind") in ("prior_run_code", "owned_by_later_wave")]
        if unclaimed:
            problems.append(f"unclaimed carryover in last wave: {unclaimed}")

# 2a. Session outcomes from the Jobs API. A session that hit the Genie Code step limit still ends its
#     for_each iteration SUCCESS; its agent response and conversation id are in runs/get-output.
#     Iteration #i runs the wave's for_each input i. API failures are warnings: 2b still applies.
gate_warnings = []


def _session_iterations():
    api = WorkspaceClient().api_client
    key = f"w{wave}_sessions"
    cands = [t for t in get_run(api, run_id).get("tasks") or [] if t.get("task_key") == key]
    if not cands:
        return {}
    fe = max(cands, key=lambda t: (int(t.get("attempt_number") or 0), int(t.get("start_time") or 0)))
    parent = get_run(api, fe["run_id"])
    raw = ((parent.get("tasks") or [{}])[0].get("for_each_task") or {}).get("inputs") or "[]"
    inputs = json.loads(raw) if isinstance(raw, str) else raw
    entries = []
    for it in parent.get("iterations") or []:
        out = get_run_output(api, it["run_id"])
        g = out.get("genie_task_output") or {}
        entries.append({"index": iteration_index((out.get("metadata") or {}).get("run_name")),
                        "attempt_number": it.get("attempt_number"), "run_id": it["run_id"],
                        "conversation_id": g.get("conversation_id"),
                        "response": (g.get("agent_output") or {}).get("response")})
    return {str(inputs[i].get("session_id")): e for i, e in latest_by_index(entries).items() if 0 <= i < len(inputs)}


try:
    iteration_info = _session_iterations()
    print(f"session iterations read from the Jobs API: {len(iteration_info)}")
except Exception as e:  # interactive run with a fake run_id, or an API error
    iteration_info = {}
    gate_warnings.append(f"session outcomes unavailable from the Jobs API ({type(e).__name__}: {str(e)[:150]}); "
                         f"status-file checks still apply")

# 2b. Independent check of every session status file (does not trust the integrator alone)
session_outcomes = {}
for s in wave_cfg["sessions"]:
    sid = str(s["session_id"])
    p = os.path.join(status_dir, f"{sid}.json")
    st = None
    if os.path.exists(p):
        with open(p) as f:
            st = json.load(f)
    it = iteration_info.get(sid) or {}
    oc = session_outcome(it.get("response"), st)
    session_outcomes[sid] = {"outcome": oc["outcome"], "conversation_id": it.get("conversation_id"),
                             "iteration_run_id": it.get("run_id")}
    if oc["warning"]:
        gate_warnings.append(f"session {sid} {oc['warning']} (conversation {it.get('conversation_id')})")
    if oc["outcome"] != "complete":
        problems.append(session_problem(sid, oc["outcome"], it.get("conversation_id"), wave,
                                        (st or {}).get("progress")))
        continue
    if st.get("status") != "pass":
        problems.append(
            f"session {s['session_id']}: status={st.get('status')!r} {st.get('blocking', '')}"
        )

# 3. Converge sessions: a 'pass' must end with a stable attempt, and files_changed must equal
#    the union of the attempts' files_changed plus files inherited from an earlier run of the
#    same session (listed in status_dir/<other_run_id>/<session_id>.json, inside owned_paths).
status_root = os.path.join(build_root, manifest["status_dir"])
inherited_by_session = {}
for s in wave_cfg["sessions"]:
    if s.get("mode") != "converge":
        continue
    p = os.path.join(status_dir, f"{s['session_id']}.json")
    if not os.path.exists(p):
        continue  # already reported above
    with open(p) as f:
        st = json.load(f)
    attempts = st.get("attempts") or []
    if st.get("status") == "pass" and (not attempts or attempts[-1].get("result") != "stable"):
        problems.append(f"session {s['session_id']}: status=pass but last attempt is not 'stable'")
    if st.get("status") == "pass":
        fc = check_files_changed(s, st, prior_run_files(status_root, run_id, s["session_id"]))
        problems.extend(fc["problems"])
        if fc["inherited"]:
            inherited_by_session[s["session_id"]] = fc["inherited"]

# The integrator should record inherited files as carryover kind "inherited" (bookkeeping only;
# a missing record is a warning, not a gate failure).
recorded = {c.get("path") for c in ((gate or {}).get("carryover_files") or []) if c.get("kind") == "inherited"}
for sid, items in inherited_by_session.items():
    print(f"session {sid}: {len(items)} inherited file(s): {items}")
    unrecorded = [i["path"] for i in items if i["path"] not in recorded]
    if gate and unrecorded:
        print(f"WARNING session {sid}: inherited files not in the gate's carryover_files: {unrecorded}")

# 4. Declared bundle validations: the integrator must report a passing
#    `bundle validate --strict` for every validate_bundles entry (roots may be relative to build_root).
def _bundle_key(root, target):
    return (os.path.normpath(os.path.join(build_root, root or ".")), target)

reported = {_bundle_key(b.get("root"), b.get("target")): b for b in ((gate or {}).get("bundle_validations") or [])}
for vb in wave_cfg.get("validate_bundles") or []:
    b = reported.get(_bundle_key(vb.get("root"), vb.get("target")))
    label = f"{vb.get('root')} --target {vb.get('target')}"
    if b is None:
        problems.append(f"validate_bundles: no result reported for {label}")
    elif b.get("result") != "pass":
        problems.append(f"validate_bundles: {label} = {b.get('result')!r} {b.get('detail', '')}")

# 5. Required checks: every wave required_checks name must be reported with result pass.
problems.extend(required_check_problems(wave_cfg, gate))

static_problems = list(problems)
print(f"static gate: {'PASS' if not static_problems else 'FAIL'} ({len(static_problems)} problems)")

In [0]:
import re
import time
from datetime import datetime, timedelta, timezone

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import ExecuteStatementRequestOnWaitTimeout, StatementParameterListItem

# app_probes.py sits next to this notebook (cwd for workspace notebooks); shared with app_probe_check.
import sys
sys.path.insert(0, os.getcwd())
from app_probes import count_mlflow_traces, probe_client, run_app_probes
from gate_helpers import genie_freshness, locate_genie_source

DEFAULT_ERROR_PATTERNS = [
    r"error TS\d+",
    r"Error building app",
    r"Unhandled",
    r"UnhandledPromiseRejection",
    r"FATAL",
    r'"level"\s*:\s*"(error|fatal)"',
]

# Lines from old-deployment shutdown that are benign during a redeploy.
# The gate's SQL query catches same-service lines after the new deployment's
# create_time (to handle runtime logs that don't carry the deployment_id attribute),
# so the old deployment's SIGTERM and ELIFECYCLE lines fall in the window.
DEFAULT_IGNORE_PATTERNS = [
    r"App did not respect SIGTERM timeout",
    r"\[ELIFECYCLE\]",
]
_TABLE_RE = re.compile(r"^[A-Za-z0-9_`\-]+\.[A-Za-z0-9_`\-]+\.[A-Za-z0-9_`\-]+$")
_ATTR_RE = re.compile(r"^[A-Za-z0-9_.\-]+$")
_KIND_RE = re.compile(r"^SPAN_KIND_[A-Z]+$")

runtime_names = wave_cfg.get("runtime_gate") or []
runtime_defs = manifest.get("runtime_targets") or {}
runtime_results = {}


def _ts(v):
    """Parse an ISO string or epoch-millis into an aware UTC datetime."""
    if v in (None, ""):
        return None
    if isinstance(v, (int, float)):
        return datetime.fromtimestamp(v / 1000, tz=timezone.utc)
    return datetime.fromisoformat(str(v).replace("Z", "+00:00"))


def _enum(v):
    return getattr(v, "value", v)


def check_app_logs(tgt, app_name, dep_id, dep_created):
    """Count error lines and check required markers for one app deployment."""
    tel = tgt.get("telemetry") or {}
    table = tel.get("logs_table")
    out = {"logs_table": table, "error_count": None, "error_samples": [], "missing_markers": []}
    if not table:
        out["note"] = "no telemetry.logs_table: log checks skipped"
        return out, []
    dep_attr = tel.get("deployment_id_attribute", "app.deployment_id")
    src_attr = tel.get("log_source_attribute", "app.log_source")
    if not _TABLE_RE.match(table) or not _ATTR_RE.match(dep_attr) or not _ATTR_RE.match(src_attr):
        return out, [f"invalid telemetry identifiers: {table!r} / {dep_attr!r} / {src_attr!r}"]

    # Rows for this deployment (build + runtime), plus same-service rows after it was created
    # in case runtime lines don't carry the deployment id attribute.
    rows = spark.sql(
        f"""
        SELECT time, severity_text, body::string AS body, attributes:['{src_attr}']::string AS src
        FROM {table}
        WHERE date >= date(:since)
          AND (attributes:['{dep_attr}']::string = :dep
               OR (service_name = :svc AND time >= :since_ts))
        ORDER BY time
        LIMIT 50000
        """,
        args={"since": dep_created.date().isoformat(), "since_ts": dep_created, "dep": dep_id, "svc": app_name},
    ).collect()

    err_res = [re.compile(p) for p in (tel.get("error_patterns") or DEFAULT_ERROR_PATTERNS)]
    ign_res = [re.compile(p) for p in (tel.get("ignore_patterns") or DEFAULT_IGNORE_PATTERNS)]
    errors = []
    for r in rows:
        body = r["body"] or ""
        if any(p.search(body) for p in ign_res):
            continue
        matched = None
        if (r["severity_text"] or "").upper() in ("ERROR", "FATAL"):
            matched = "severity"
        else:
            for p in err_res:
                if p.search(body):
                    matched = f"pattern:{p.pattern}"
                    break
        if matched:
            errors.append(f"{r['time']} [{matched}] [{r['src'] or '?'}] {body[:200]}")

    runtime_rows = [r for r in rows if (r["src"] or "").upper() != "BUILD"]
    missing = [m for m in (tel.get("required_markers") or [])
               if not any(re.search(m, r["body"] or "") for r in runtime_rows)]

    out.update({"rows": len(rows), "runtime_rows": len(runtime_rows), "error_count": len(errors),
                "error_samples": errors[:10], "missing_markers": missing})
    max_errors = int((tgt.get("stability") or {}).get("max_errors", 0))
    probs = []
    if len(errors) > max_errors:
        hint = " (add ignore_patterns for benign lines)" if any("[severity]" in e for e in errors) else ""
        probs.append(f"{len(errors)} error log lines (max {max_errors}){hint}; first: {errors[0] if errors else ''}")
    if missing:
        probs.append(f"required log markers not found: {missing}")
    return out, probs


def check_app(tgt, run_start):
    name = tgt["name"]
    soak = int((tgt.get("stability") or {}).get("soak_minutes", 10))
    fresh = tgt.get("require_fresh_deployment", True)
    info, probs = {"app": name}, []

    app = w.apps.get(name)
    ad, pd = app.active_deployment, app.pending_deployment
    if pd is not None and _enum(getattr(pd.status, "state", None)) == "IN_PROGRESS":
        probs.append(f"pending deployment {pd.deployment_id} still IN_PROGRESS")
    if ad is None:
        return info, probs + ["no active deployment"]
    dep_id, dep_created = ad.deployment_id, _ts(ad.create_time)
    dstate = _enum(getattr(ad.status, "state", None))
    info.update({"deployment_id": dep_id, "deployment_state": dstate, "deployment_created": str(dep_created)})
    if dstate != "SUCCEEDED":
        probs.append(f"active deployment {dep_id} is {dstate}: {getattr(ad.status, 'message', '')}")
    if fresh and run_start and dep_created and dep_created < run_start:
        probs.append(f"active deployment {dep_id} ({dep_created}) predates this build run ({run_start})")

    def _states(a):
        return _enum(getattr(a.app_status, "state", None)), _enum(getattr(a.compute_status, "state", None))

    app_state, compute_state = _states(app)
    info.update({"app_state": app_state, "compute_state": compute_state})
    if app_state != "RUNNING" or compute_state != "ACTIVE":
        probs.append(f"app_status={app_state} compute_status={compute_state}: {getattr(app.app_status, 'message', '')}")

    # Probes run at the start of the soak, so min_traces counts the spans they caused.
    probe_time = None
    if not probs and tgt.get("probes"):
        probe_time = datetime.now(timezone.utc)
        try:
            pw, info["probe_identity"] = probe_client(w, tgt.get("probe_auth") or manifest.get("probe_auth"),
                                                      dbutils.secrets.get)
            probe_results, probe_probs = run_app_probes(w, name, tgt["probes"], probe_w=pw)
        except Exception as e:  # missing secret scope/key
            probe_results, probe_probs = [], [f"probe_auth: {type(e).__name__}: {str(e)[:200]}"]
        info["probes"] = probe_results
        probs += probe_probs

    # Soak only when everything so far is healthy
    if not probs and soak > 0:
        print(f"  {name}: soaking {soak} min on deployment {dep_id} ...")
        deadline = time.time() + soak * 60
        while time.time() < deadline:
            time.sleep(60)
            a = w.apps.get(name)
            s_app, s_comp = _states(a)
            cur_dep = a.active_deployment.deployment_id if a.active_deployment else None
            if s_app != "RUNNING" or s_comp != "ACTIVE" or cur_dep != dep_id:
                probs.append(f"unstable during soak: app_status={s_app} compute={s_comp} deployment={cur_dep}")
                break
        info["soak_minutes"] = soak

    if dep_created:
        log_info, log_probs = check_app_logs(tgt, name, dep_id, dep_created)
        info["logs"] = log_info
        probs += log_probs

    stab = tgt.get("stability") or {}
    if (stab.get("min_traces") is not None or stab.get("max_trace_errors") is not None) and not probs:
        tr_info, tr_probs = check_app_traces(tgt, name, probe_time or dep_created)
        info["traces"] = tr_info
        probs += tr_probs
    return info, probs


def _otel_trace_counts(tel, app_name, since):
    """OTel spans table: distinct traces and error traces for the app since `since`.

    Spans carry no deployment id, so scoping is service_name + time. Only SERVER spans count by default:
    apps also emit periodic CLIENT spans (their own telemetry export) that would satisfy any floor.
    Error traces = any span (any kind) with status.code STATUS_CODE_ERROR.
    """
    table = tel["traces_table"]
    svc = tel.get("trace_service_name") or app_name
    kinds = tel.get("trace_span_kinds", ["SPAN_KIND_SERVER"])
    if not _TABLE_RE.match(table) or not all(_KIND_RE.match(k) for k in kinds or []):
        raise ValueError(f"invalid traces config: table={table!r} span_kinds={kinds!r}")
    kind_expr = f"kind IN ({', '.join(repr(k) for k in kinds)})" if kinds else "TRUE"
    row = spark.sql(
        f"""
        SELECT count(DISTINCT CASE WHEN {kind_expr} THEN trace_id END) AS traces,
               count(DISTINCT CASE WHEN status.code = 'STATUS_CODE_ERROR' THEN trace_id END) AS errors,
               max(time) AS last_time
        FROM {table}
        WHERE date >= date(:since) AND time >= :since_ts AND service_name = :svc
        """,
        args={"since": since.date().isoformat(), "since_ts": since, "svc": svc},
    ).collect()[0]
    return {"source": "otel", "traces_table": table, "service_name": svc, "span_kinds": kinds or "all",
            "traces": row["traces"], "errors": row["errors"], "last_time": str(row["last_time"])}


def check_app_traces(tgt, app_name, since):
    """Trace floor / error budget since `since` (probe time, else deployment time).

    Sources: telemetry.traces_table (OTel spans in UC) and/or telemetry.mlflow_experiment_id (MLflow 3
    traces). With both, each must meet min_traces and the error budget applies to each.
    """
    tel = tgt.get("telemetry") or {}
    stab = tgt.get("stability") or {}
    min_traces = stab.get("min_traces")
    max_err = stab.get("max_trace_errors")
    wait_min = int(stab.get("traces_wait_minutes", 5))
    out = {"since": str(since), "min_traces": min_traces, "max_trace_errors": max_err, "sources": []}
    if since is None:
        return out, ["no probe or deployment time to count traces from"]
    readers = []
    if tel.get("traces_table"):
        readers.append(lambda: _otel_trace_counts(tel, app_name, since))
    if tel.get("mlflow_experiment_id"):
        def _ml():
            r = count_mlflow_traces(tel["mlflow_experiment_id"], int(since.timestamp() * 1000),
                                    tel.get("mlflow_trace_filter"))
            return {"source": "mlflow", "experiment_id": str(tel["mlflow_experiment_id"]), **r}
        readers.append(_ml)
    if not readers:
        return out, ["min_traces / max_trace_errors need telemetry.traces_table or telemetry.mlflow_experiment_id"]

    deadline = time.time() + wait_min * 60
    while True:
        try:
            results = [r() for r in readers]
        except Exception as e:
            return out, [f"trace query failed: {type(e).__name__}: {str(e)[:200]}"]
        enough = min_traces is None or all(x["traces"] >= int(min_traces) for x in results)
        if enough or time.time() > deadline:
            break
        time.sleep(30)  # spans / traces land with ingestion lag
    out["sources"] = results
    probs = []
    for x in results:
        where = x.get("traces_table") or f"experiment {x.get('experiment_id')}"
        if min_traces is not None and x["traces"] < int(min_traces):
            probs.append(f"{x['traces']} traces in {where} since {since} (min {min_traces})")
        if max_err is not None and x["errors"] > int(max_err):
            probs.append(f"{x['errors']} error traces in {where} since {since} (max {max_err})")
    return out, probs


def check_job(tgt, run_start):
    info, probs = {}, []
    job_id = tgt.get("id")
    if not job_id:
        matches = list(w.jobs.list(name=tgt["name"]))
        if len(matches) != 1:
            return info, [f"job name {tgt['name']!r} matched {len(matches)} jobs; set runtime target 'id'"]
        job_id = matches[0].job_id
    info["job_id"] = job_id
    soak = int((tgt.get("stability") or {}).get("soak_minutes", 10))
    start_ms = int(run_start.timestamp() * 1000) if run_start else None
    deadline = time.time() + soak * 60
    while True:
        runs = list(w.jobs.list_runs(job_id=int(job_id), start_time_from=start_ms, limit=5))
        if not runs:
            return info, ["no job run started since the build run began"]
        latest = runs[0]
        life = _enum(getattr(latest.state, "life_cycle_state", None))
        if life in ("TERMINATED", "SKIPPED", "INTERNAL_ERROR") or time.time() > deadline:
            break
        time.sleep(30)
    result = _enum(getattr(latest.state, "result_state", None))
    info.update({"run_id": latest.run_id, "life_cycle_state": life, "result_state": result})
    if result != "SUCCESS":
        probs.append(f"latest run {latest.run_id}: {life}/{result} {getattr(latest.state, 'state_message', '')}")
    return info, probs


def check_pipeline(tgt, run_start):
    info, probs = {}, []
    pid = tgt.get("id")
    if not pid:
        matches = [p for p in w.pipelines.list_pipelines() if p.name == tgt["name"]]
        if len(matches) != 1:
            return info, [f"pipeline name {tgt['name']!r} matched {len(matches)} pipelines; set runtime target 'id'"]
        pid = matches[0].pipeline_id
    info["pipeline_id"] = pid
    soak = int((tgt.get("stability") or {}).get("soak_minutes", 10))
    start_ms = int(run_start.timestamp() * 1000) if run_start else 0
    deadline = time.time() + soak * 60
    while True:
        ups = [u for u in (w.pipelines.list_updates(pipeline_id=pid, max_results=10).updates or [])
               if (u.creation_time or 0) >= start_ms]
        if not ups:
            return info, ["no pipeline update started since the build run began"]
        latest = max(ups, key=lambda u: u.creation_time or 0)
        state = _enum(latest.state)
        if state in ("COMPLETED", "FAILED", "CANCELED") or time.time() > deadline:
            break
        time.sleep(30)
    info.update({"update_id": latest.update_id, "state": state})
    if state == "RUNNING":
        info["note"] = "continuous pipeline stayed RUNNING through the soak"
    elif state != "COMPLETED":
        probs.append(f"latest update {latest.update_id} is {state}")

    stab = tgt.get("stability") or {}
    limits = {k: stab[k] for k in ("max_failed_records", "max_dropped_records") if stab.get(k) is not None}
    if limits:
        try:
            dq = _pipeline_data_quality(pid, latest.update_id)
        except Exception as e:
            return info, probs + [f"data quality metrics unreadable: {type(e).__name__}: {str(e)[:200]}"]
        info["data_quality"] = dq
        top = ", ".join(f"{k}={v}" for k, v in list(dq["failing_expectations"].items())[:5])
        if "max_failed_records" in limits and dq["failed_records"] > int(limits["max_failed_records"]):
            probs.append(f"{dq['failed_records']} failed expectation records > max_failed_records "
                         f"{limits['max_failed_records']} ({top})")
        if "max_dropped_records" in limits and dq["dropped_records"] > int(limits["max_dropped_records"]):
            probs.append(f"{dq['dropped_records']} dropped records > max_dropped_records {limits['max_dropped_records']}")
    return info, probs


_UUID_RE = re.compile(r"^[0-9a-fA-F-]{36}$")


def _pipeline_data_quality(pid, update_id):
    """Sum flow_progress data_quality metrics for one update. Reads the pipeline's Unity Catalog event log
    table when the pipeline publishes one (spec.event_log), else the events API. Batch flows report once,
    on COMPLETED; streaming flows report per micro-batch, so summing is correct for both."""
    if not _UUID_RE.match(update_id or ""):
        raise ValueError(f"unexpected update id {update_id!r}")
    spec = w.pipelines.get(pid).spec
    el = getattr(spec, "event_log", None)
    if el and el.name:
        parts = [el.catalog or spec.catalog, el.schema or spec.schema or spec.target, el.name]
        table = ".".join(f"`{p}`" for p in parts)
        rows = spark.sql(
            f"""SELECT details:flow_progress.data_quality::string AS dq FROM {table}
                WHERE event_type = 'flow_progress' AND origin.update_id = :u
                  AND details:flow_progress.data_quality IS NOT NULL""", args={"u": update_id}).collect()
        dqs, source = [json.loads(r.dq) for r in rows], f"event log table {'.'.join(parts)}"
    else:
        dqs, source = [], "pipeline events API"
        for e in w.pipelines.list_pipeline_events(pid, filter=f"update_id = '{update_id}'", max_results=250):
            if e.event_type != "flow_progress":  # the API filter does not support event_type
                continue
            d_ = ((e.as_dict().get("details") or {}).get("flow_progress") or {}).get("data_quality")
            if d_:
                dqs.append(d_)
    failing = {}
    for d_ in dqs:
        for x in d_.get("expectations") or []:
            if x.get("failed_records"):
                k = f"{x.get('dataset', '?').split('.')[-1]}.{x.get('name')}"
                failing[k] = failing.get(k, 0) + int(x["failed_records"])
    failing = dict(sorted(failing.items(), key=lambda kv: -kv[1]))
    return {"source": source, "events": len(dqs),
            "dropped_records": sum(int(d_.get("dropped_records") or 0) for d_ in dqs),
            "warned_records": sum(int(d_.get("warned_records") or 0) for d_ in dqs),
            "failed_records": sum(failing.values()), "failing_expectations": failing}


_NO_DATA_COMMITS = ("VIEW", "METRIC_VIEW", "FOREIGN")  # no Delta history or refresh time


def _table_last_write(fq, ttype):
    """(datetime | None, source). Delta / streaming tables: last commit; materialized views: last refresh."""
    quoted = ".".join(f"`{p}`" for p in fq.split("."))
    if ttype == "MATERIALIZED_VIEW":
        meta = json.loads(spark.sql(f"DESCRIBE EXTENDED {quoted} AS JSON").first()[0])
        return _ts((meta.get("refresh_information") or {}).get("last_refreshed_at")), "last refresh"
    row = spark.sql(f"DESCRIBE HISTORY {quoted} LIMIT 1").selectExpr("unix_millis(timestamp) AS ms", "operation").first()
    return (_ts(row.ms) if row else None), f"last commit ({row.operation if row else 'none'})"


def check_table(tgt, run_start):
    """Exists in UC, has required_columns, at least min_rows rows, and was written during this run."""
    info, probs = {}, []
    fq = tgt["name"]
    try:
        t = w.tables.get(fq)
    except Exception as e:
        return info, [f"{fq} does not resolve in Unity Catalog ({type(e).__name__})"]
    ttype = _enum(t.table_type)
    cols = {c.name.lower() for c in (t.columns or [])}
    info.update({"table_type": ttype, "columns": len(cols)})
    missing = [c for c in tgt.get("required_columns") or [] if c.lower() not in cols]
    if missing:
        probs.append(f"missing columns: {missing}")
    if tgt.get("min_rows") is not None:
        quoted = ".".join(f"`{p}`" for p in fq.split("."))
        try:
            n = spark.sql(f"SELECT count(*) AS n FROM {quoted}").first().n
            info["rows"] = n
            if n < int(tgt["min_rows"]):
                probs.append(f"{n} rows < min_rows {tgt['min_rows']}")
        except Exception as e:
            probs.append(f"row count failed: {str(e).splitlines()[0][:200]}")
    if tgt.get("require_fresh_deployment", True):
        if ttype in _NO_DATA_COMMITS:
            info["freshness"] = f"skipped: {ttype} has no data commits"
        else:
            try:
                last, src = _table_last_write(fq, ttype)
                info.update({"last_write": str(last), "last_write_source": src})
                if run_start and (last is None or last < run_start):
                    probs.append(f"{src} {last} predates this build run ({run_start}): not written by this build")
            except Exception as e:
                probs.append(f"freshness check failed: {str(e).splitlines()[0][:200]}")
    return info, probs


_PARAM_TYPES = {"INTEGER": "INT", "DATETIME": "TIMESTAMP"}  # Lakeview dataType -> SQL parameter type
_SQL_TERMINAL = ("SUCCEEDED", "FAILED", "CANCELED", "CLOSED")


def _run_sql(stmt, warehouse_id, timeout_s, catalog=None, schema=None, parameters=None):
    """Run one statement on a warehouse. Returns (state, error, first two result rows as text)."""
    r = w.statement_execution.execute_statement(
        statement=stmt, warehouse_id=warehouse_id, catalog=catalog, schema=schema, parameters=parameters,
        wait_timeout="30s", on_wait_timeout=ExecuteStatementRequestOnWaitTimeout.CONTINUE)
    deadline = time.time() + timeout_s
    while _enum(r.status.state) not in _SQL_TERMINAL:
        if time.time() > deadline:
            w.statement_execution.cancel_execution(r.statement_id)
            return "TIMEOUT", f"no result after {timeout_s}s", None
        time.sleep(3)
        r = w.statement_execution.get_statement(r.statement_id)
    err = r.status.error.message if r.status.error else None
    rows = (r.result.data_array or []) if r.result else []
    head = " ".join(str(x[0]) for x in rows[:2] if x and x[0])
    return _enum(r.status.state), err, head


def _dataset_params(ds):
    """Single-value dataset parameters -> statement parameters (defaults). Multi-value/range ones are unsupported."""
    items, unsupported = [], []
    for p in ds.get("parameters") or []:
        vals = (((p.get("defaultSelection") or {}).get("values") or {}).get("values")) or []
        if len(vals) == 1 and "value" in vals[0]:
            t = (p.get("dataType") or "STRING").upper()
            items.append(StatementParameterListItem(name=p["keyword"], value=str(vals[0]["value"]), type=_PARAM_TYPES.get(t, t)))
        else:
            unsupported.append(p.get("keyword"))
    return items, unsupported


def check_dashboard(tgt, run_start):
    """Published in this run, warehouse set, every dataset resolves (EXPLAIN by default, or LIMIT 1 execution)."""
    info, probs = {}, []
    did = tgt.get("id")
    if not did:
        matches = [d for d in w.lakeview.list() if d.display_name == tgt["name"]]
        if len(matches) != 1:
            return info, [f"dashboard name {tgt['name']!r} matched {len(matches)} dashboards; set runtime target 'id'"]
        did = matches[0].dashboard_id
    info["dashboard_id"] = did
    try:
        pub = w.lakeview.get_published(did)
    except Exception as e:
        return info, [f"dashboard is not published ({type(e).__name__})"]
    rev = _ts(pub.revision_create_time)
    info.update({"published_revision": str(rev), "embed_credentials": pub.embed_credentials})
    if tgt.get("require_fresh_deployment", True) and run_start and rev and rev < run_start:
        probs.append(f"published revision ({rev}) predates this build run ({run_start})")

    # Bundle deploy publishes the deployed draft, so the draft's datasets are what viewers run.
    draft = w.lakeview.get(did)
    wh = tgt.get("warehouse_id") or pub.warehouse_id or draft.warehouse_id
    if not wh:
        return info, probs + ["no warehouse configured on the dashboard or the runtime target"]
    info["warehouse_id"] = wh
    datasets = json.loads(draft.serialized_dashboard or "{}").get("datasets") or []
    if not datasets:
        probs.append("dashboard has no datasets")
    execute = bool(tgt.get("execute_datasets", False))
    timeout_s = int(tgt.get("dataset_timeout_seconds", 120))
    info["dataset_check"] = "execute (LIMIT 1)" if execute else "explain"

    results = []
    for ds in datasets:
        label = ds.get("displayName") or ds.get("name")
        cat, sch = ds.get("catalog"), ds.get("schema")
        if ds.get("asset_name"):
            # Table / metric view dataset: it must resolve in Unity Catalog
            a = ds["asset_name"]
            full = a if a.count(".") == 2 else ".".join(x for x in (cat, sch, a) if x)
            try:
                w.tables.get(full)
                results.append({"dataset": label, "asset": full, "result": "pass"})
            except Exception as e:
                results.append({"dataset": label, "asset": full, "result": "fail", "detail": str(e)[:200]})
                probs.append(f"dataset {label!r}: asset {full} does not resolve")
            continue
        query = "".join(ds.get("queryLines") or []).strip().rstrip(";").strip()
        if not query:
            results.append({"dataset": label, "result": "skipped", "detail": "no query"})
            continue
        params, unsupported = _dataset_params(ds)
        if unsupported:
            results.append({"dataset": label, "result": "skipped",
                            "detail": f"multi-value/range parameters not supported: {unsupported}"})
            continue
        stmt = f"SELECT * FROM ({query}) LIMIT 1" if execute else f"EXPLAIN {query}"
        state, err, head = _run_sql(stmt, wh, timeout_s, cat, sch, params or None)
        bad = None
        if state != "SUCCEEDED":
            bad = f"{state}: {(err or '')[:200]}"
        elif not execute and head and "Error occurred during query planning" in head:
            # EXPLAIN reports analysis errors as a SUCCEEDED row, not a failed statement
            bad = head.strip()[:300]
        results.append({"dataset": label, "result": "fail" if bad else "pass", **({"detail": bad} if bad else {})})
        if bad:
            probs.append(f"dataset {label!r}: {bad}")
    info["datasets"] = results
    if any(r["result"] == "skipped" for r in results):
        info["note"] = "some datasets were skipped; see datasets"
    return info, probs


def _genie_space_id(tgt):
    """Space titles are not unique: a name must match exactly one space."""
    if tgt.get("id"):
        return tgt["id"], None
    matches, token = [], None
    while True:
        r = w.genie.list_spaces(page_size=100, page_token=token)
        matches += [s.space_id for s in (r.spaces or []) if s.title == tgt["name"]]
        token = r.next_page_token
        if not token:
            break
    if len(matches) != 1:
        return None, f"Genie space title {tgt['name']!r} matched {len(matches)} spaces; set runtime target 'id'"
    return matches[0], None


def _genie_space_freshness(tgt, run_start, upd, space):
    """fresh (updated after the run started) or current (deployed content equals the bundle source).
    A no-op deploy leaves update_time alone, so content is compared before failing on the timestamp."""
    after = (upd >= run_start) if upd else None
    src, basis = None, None
    if not after:
        path, basis = locate_genie_source(tgt, build_root)
        if path:
            try:
                with open(path) as f:
                    src = json.load(f)
                basis = f"{basis} ({os.path.relpath(path, build_root)})"
            except Exception as e:
                src, basis = None, f"source file {path} unreadable ({type(e).__name__})"
    fr = genie_freshness(after, src, basis, space)
    if fr["result"] == "pass":
        return fr, []
    msg = f"freshness {fr['basis']}: space last updated {upd}, before this build run ({run_start})" if upd else (
        f"freshness {fr['basis']}: space has no update_time")
    if fr.get("differences"):
        msg += f"; first differences: {fr['differences'][:5]}"
    if fr["basis"].startswith("timestamp only"):
        msg += " (set source_file or resource_key to compare content, or require_fresh_deployment: false)"
    return fr, [msg]


def check_genie_space(tgt, run_start):
    """Updated in this run or content equal to the bundle source, warehouse exists, every table /
    metric view and configured column resolves, example SQL passes EXPLAIN. Then optional
    probe_questions (warn by default); they run even when only freshness failed, as advisory results."""
    info, probs = {}, []
    sid, err = _genie_space_id(tgt)
    if err:
        return info, [err]
    info["space_id"] = sid
    try:
        # The SDK model drops create_time / update_time, so read the raw response.
        raw = w.api_client.do("GET", f"/api/2.0/genie/spaces/{sid}")
    except Exception as e:
        return info, [f"Genie space {sid} not readable ({type(e).__name__}: {str(e)[:150]})"]
    upd = _ts(raw.get("update_time"))
    info.update({"title": raw.get("title"), "update_time": str(upd) if upd else None})
    fresh_probs = []  # kept apart so probe questions still run when only freshness failed

    wh = raw.get("warehouse_id")
    info["warehouse_id"] = wh
    if not wh:
        probs.append("no warehouse attached to the space")
    else:
        if tgt.get("warehouse_id") and tgt["warehouse_id"] != wh:
            probs.append(f"space uses warehouse {wh}, runtime target expects {tgt['warehouse_id']}")
        try:
            info["warehouse_state"] = _enum(w.warehouses.get(wh).state)
        except Exception as e:
            probs.append(f"warehouse {wh} not found or not accessible ({type(e).__name__})")
            wh = None

    try:
        space = json.loads(w.genie.get_space(sid, include_serialized_space=True).serialized_space or "{}")
    except Exception as e:
        return info, probs + [f"cannot read the space configuration ({type(e).__name__}); "
                              f"the build job's identity needs CAN_EDIT on the space"]
    if tgt.get("require_fresh_deployment", True) and run_start:
        info["freshness"], fresh_probs = _genie_space_freshness(tgt, run_start, upd, space)
    else:
        info["freshness"] = {"result": "skipped", "basis": "require_fresh_deployment: false" if run_start
                             else "build run start time unknown"}

    tables = (space.get("data_sources") or {}).get("tables") or []
    if not tables:
        probs.append("space has no tables or metric views")
    results = []
    for t in tables:
        full = t.get("identifier", "")
        entry = {"table": full}
        try:
            ti = w.tables.get(full)
            entry["type"] = _enum(ti.table_type)
            # column_configs left behind after a table changed (renamed/dropped columns)
            cols = {c.name.lower() for c in (ti.columns or [])}
            missing = [c["column_name"] for c in (t.get("column_configs") or [])
                       if c.get("column_name") and c["column_name"].lower() not in cols]
            if missing:
                entry.update(result="fail", detail=f"configured columns not in table: {missing}")
                probs.append(f"{full}: configured columns not in table: {missing}")
            else:
                entry["result"] = "pass"
        except Exception as e:
            entry.update(result="fail", detail=f"{type(e).__name__}: {str(e)[:150]}")
            probs.append(f"{full} does not resolve in Unity Catalog")
        results.append(entry)
    info["tables"] = results

    instr = space.get("instructions") or {}
    ex_results = []
    if tgt.get("check_example_sql", True) and wh:
        timeout_s = int(tgt.get("sql_timeout_seconds", 120))
        for ex in instr.get("example_question_sqls") or []:
            q = (ex.get("question") or [""])[0][:80]
            sql = "".join(ex.get("sql") or []).strip().rstrip(";").strip()
            if not sql:
                continue
            if ex.get("parameters"):
                ex_results.append({"question": q, "result": "skipped", "detail": "parameterized"})
                continue
            state, err, head = _run_sql(f"EXPLAIN {sql}", wh, timeout_s)
            bad = f"{state}: {(err or '')[:200]}" if state != "SUCCEEDED" else (
                head.strip()[:300] if head and "Error occurred during query planning" in head else None)
            ex_results.append({"question": q, "result": "fail" if bad else "pass", **({"detail": bad} if bad else {})})
            if bad:
                probs.append(f"example SQL {q!r}: {bad}")
    info["example_sql"] = ex_results
    # SQL snippets (measures, filters) are fragments without table context: counted, not checked.
    info["sql_snippets_not_checked"] = sum(len(v or []) for v in (instr.get("sql_snippets") or {}).values())

    questions = tgt.get("probe_questions") or []
    if questions:
        mode = tgt.get("probe_mode", "warn")
        if probs:
            info["probe_questions"] = [{"question": q, "result": "skipped", "detail": "static checks failed"}
                                       for q in questions]
        else:
            timeout_s = int(tgt.get("probe_timeout_seconds", 120))
            q_results = []
            for q in questions:
                res = _ask_genie(sid, q, timeout_s)
                if res["result"] != "pass":  # one retry, in a new conversation
                    res = dict(_ask_genie(sid, q, timeout_s), first_attempt=res.get("detail"))
                q_results.append(res)
            info["probe_questions"] = q_results
            q_probs = [f"question {r['question']!r}: {r.get('detail')}" for r in q_results if r["result"] != "pass"]
            if mode == "fail" and not fresh_probs:
                probs += q_probs
            elif q_probs:
                # warn mode, or advisory because freshness already fails the target
                info["warnings"] = q_probs
            if fresh_probs:
                info["probe_note"] = "freshness failed: probe results are advisory"
        info["probe_mode"] = mode
    return info, fresh_probs + probs


def _ask_genie(sid, question, timeout_s):
    """Ask one question in a new conversation. Pass = Genie produced SQL and the SQL ran.
    Never judges the answer: LLM output is not deterministic. The conversation is left in the
    build identity's Genie history."""
    res = {"question": question[:200], "result": "fail"}
    t0 = time.time()
    try:
        m = w.genie.start_conversation_and_wait(sid, question, timeout=timedelta(seconds=timeout_s))
    except Exception as e:  # TimeoutError, or the space rejected the request
        res.update(seconds=round(time.time() - t0, 1), detail=f"{type(e).__name__}: {str(e)[:200]}")
        return res
    res.update(seconds=round(time.time() - t0, 1), status=_enum(m.status), conversation_id=m.conversation_id)
    if res["status"] != "COMPLETED":
        res["detail"] = f"message {res['status']}: {str(m.error)[:200] if m.error else ''}"
        return res
    qa = next((a for a in (m.attachments or []) if a.query), None)
    if not qa:
        text = next((a.text.content for a in (m.attachments or []) if a.text and a.text.content), "")
        res["detail"] = f"no SQL generated: {text[:200]}"
        return res
    res["sql"] = (qa.query.query or "")[:500]
    try:
        sr = w.genie.get_message_attachment_query_result(sid, m.conversation_id, m.message_id,
                                                         qa.attachment_id).statement_response
    except Exception as e:
        res["detail"] = f"query result unavailable: {type(e).__name__}: {str(e)[:200]}"
        return res
    res["statement_state"] = _enum(sr.status.state)
    if res["statement_state"] != "SUCCEEDED":
        err = sr.status.error.message if sr.status.error else ""
        res["detail"] = f"query {res['statement_state']}: {err[:200]}"
        return res
    res["rows"] = sr.manifest.total_row_count if sr.manifest else None
    res["result"] = "pass"
    return res


if not runtime_names:
    print("no runtime_gate targets for this wave")
elif static_problems:
    print("static gate failed: runtime checks skipped")
    runtime_results = {n: {"skipped": "static gate failed"} for n in runtime_names}
else:
    w = WorkspaceClient()
    try:
        run_start = _ts(w.jobs.get_run(int(run_id)).start_time)
    except Exception as e:  # interactive run with a fake run_id
        print(f"WARN: could not read build run start time ({e}); freshness check disabled")
        run_start = None
    checkers = {"app": check_app, "job": check_job, "pipeline": check_pipeline, "dashboard": check_dashboard,
                "genie_space": check_genie_space, "table": check_table}
    for n in runtime_names:
        tgt = runtime_defs.get(n)
        if not tgt:
            problems.append(f"runtime target {n!r}: not defined in manifest runtime_targets")
            continue
        fn = checkers.get(tgt.get("type"))
        if not fn:
            problems.append(f"runtime target {n!r}: unsupported type {tgt.get('type')!r}")
            continue
        try:
            info, probs = fn(tgt, run_start)
        except Exception as e:
            info, probs = {}, [f"check raised {type(e).__name__}: {e}"]
        info["status"] = "pass" if not probs else "fail"
        info["problems"] = probs
        runtime_results[n] = info
        problems += [f"runtime {n}: {p}" for p in probs]
        print(f"  runtime {n} ({tgt.get('type')}): {info['status'].upper()} {probs or ''}")
        for wmsg in info.get("warnings") or []:
            print(f"    WARN (does not fail the wave): {wmsg}")

if runtime_names:
    os.makedirs(status_dir, exist_ok=True)
    with open(os.path.join(status_dir, f"wave_{wave}_runtime.json"), "w") as f:
        json.dump({"wave": wave, "run_id": run_id, "targets": runtime_results}, f, indent=2, default=str)

In [0]:
if gate:
    skipped = [c["name"] for c in gate.get("checks", []) if c.get("result") == "skipped"]
    print(f"commit_sha: {gate.get('commit_sha') or '(none)'}")
    print(f"skipped checks: {skipped or 'none'}")
    if gate.get("suppressions"):
        print(f"suppressions to review: {json.dumps(gate['suppressions'], indent=2)}")
    if gate.get("requests"):
        print(f"requests for next wave / user: {json.dumps(gate['requests'], indent=2)}")
    if gate.get("carryover_files"):
        print(f"carryover from earlier runs (not strays): {json.dumps(gate['carryover_files'], indent=2)}")

passed = len(problems) == 0

# Set task value for the downstream If/else condition task.
# The condition_task checks: {{tasks.wN_gate.values.gate_passed}} == "true"
# If false, all downstream wave tasks are Excluded (zero-compute cascade stop).
try:
    dbutils.jobs.taskValues.set(key="gate_passed", value="true" if passed else "false")
except Exception:
    pass  # interactive run — no job context, taskValues unavailable

result = {
    "wave": wave,
    "status": "pass" if passed else "fail",
    "commit_sha": (gate or {}).get("commit_sha", ""),
    "runtime": {n: r.get("status", r.get("skipped")) for n, r in runtime_results.items()},
}
runtime_warnings = {n: r["warnings"] for n, r in runtime_results.items() if r.get("warnings")}
if runtime_warnings:  # e.g. Genie probe questions in warn mode
    result["warnings"] = runtime_warnings
if gate_warnings:  # Jobs API unavailable, step limit after the final status, ...
    result["gate_warnings"] = gate_warnings
    for wmsg in gate_warnings:
        print(f"WARN (does not fail the wave): {wmsg}")
result["sessions"] = session_outcomes
if problems:
    result["problems"] = problems
    print(gate_failure_message(wave, problems))
else:
    print(f"Wave {wave} gate PASSED ({len(wave_cfg['sessions'])} sessions, {len(runtime_results)} runtime targets).")

# The run-level build_verdict task reads this file (and the task state) and fails the run.
payload = json.dumps(result, indent=2, default=str)
print(payload)
try:
    os.makedirs(status_dir, exist_ok=True)
    with open(gate_result_path(status_dir, wave), "w") as f:
        f.write(payload)
except OSError as e:
    print(f"WARN: could not write {gate_result_path(status_dir, wave)}: {e}")

# Exit with the payload but do NOT raise. Raising would mark the next wave UPSTREAM_FAILED, and its
# run_if ALL_DONE integrator and gate would still run. The If/else task Excludes later waves (zero
# compute); build_verdict fails the run.
dbutils.notebook.exit(json.dumps(result, default=str))